# Cryogenic — Local Fedora / AMD Renderer

Same offline music visualizer as `Cryogenic_Render.ipynb`, but set up to run on a local Linux box with Mesa OpenGL (e.g. Fedora + RX 6700 via RadeonSI). No NVIDIA bits, no Colab uploads.

**Before you start** (one-time, in a regular shell — not in the notebook, since dnf needs sudo):

```bash
sudo dnf install -y mesa-libEGL mesa-libGL mesa-dri-drivers ffmpeg python3-pip git
```

Then in the directory you want to work in:

```bash
git clone https://github.com/splice11/music-viz.git
cd music-viz
python3 -m venv .venv
source .venv/bin/activate
pip install moderngl numpy notebook
jupyter notebook colab_render/Cryogenic_Render_Local.ipynb
```

If `dnf install ffmpeg` fails on a fresh Fedora, enable RPM Fusion first: https://rpmfusion.org/Configuration

## 1. Verify GPU rendering

On Fedora the system Mesa ICD lives at `/usr/share/glvnd/egl_vendor.d/50_mesa.json`. The renderer will iterate EGL devices and pick the first hardware one — for an RX 6700 you should see `AMD Radeon …` or `radeonsi` in `GL_RENDERER`. If you see `llvmpipe`, rendering will be ~1 fps; check that `mesa-dri-drivers` is installed.

In [ ]:
import os, sys, glob, pathlib
REPO = str(pathlib.Path.cwd().resolve().parents[0]) if pathlib.Path.cwd().name == 'colab_render' else str(pathlib.Path.cwd().resolve())
sys.path.insert(0, REPO)
print('repo root:', REPO)

print('\n--- EGL ICDs present ---')
for p in sorted(glob.glob('/usr/share/glvnd/egl_vendor.d/*.json')):
    print(' ', p)

# Drop any cached modules so a fresh import picks up the current renderer.
for m in list(sys.modules):
    if m.startswith('moderngl') or m.startswith('colab_render'):
        del sys.modules[m]

from colab_render.renderer import _make_ctx
ctx = _make_ctx(verbose=True)
print('\nGL_RENDERER:', ctx.info.get('GL_RENDERER'))
print('GL_VERSION :', ctx.info.get('GL_VERSION'))
assert 'llvmpipe' not in ctx.info.get('GL_RENDERER','').lower(), \
    'Software rendering selected — install mesa-dri-drivers and re-check.'
ctx.release()
print('\nReady to render on GPU.')

## 2. Point at your audio + analysis JSON

Edit the two paths below to match your files. The repo ships with `viz_data_v2.json` at the root — set that as `JSON_PATH` if you're rendering the bundled song.

In [ ]:
AUDIO_PATH = '/path/to/your_song.mp3'   # mp3, wav, m4a, …
JSON_PATH  = '/path/to/viz_data_v2.json'

import os
assert os.path.exists(AUDIO_PATH), f'audio not found: {AUDIO_PATH}'
assert os.path.exists(JSON_PATH),  f'json not found: {JSON_PATH}'
print('Audio:', AUDIO_PATH)
print('JSON :', JSON_PATH)

## 3. Render

Mode 2 (Kerr-Newman black hole) is heavy — at 1080p it's the slowest section by far. Start with **720p** for a sanity-check pass; bump to 1080p once the output looks right.

In [ ]:
WIDTH       = 1280   # 1920 for full 1080p
HEIGHT      = 720    # 1080 for full 1080p
TARGET_FPS  = 60
OUTPUT_PATH = os.path.join(REPO, 'cryogenic.mp4')

import sys
for m in list(sys.modules):
    if m.startswith('colab_render'):
        del sys.modules[m]
import colab_render

print('Building feature timeline…')
fd = colab_render.build(JSON_PATH, target_fps=TARGET_FPS)
print(f'  {fd.n_frames} frames @ {fd.fps:.0f} fps  ({fd.duration:.1f} s)')

print('Rendering…')
colab_render.render_to_video(
    fd,
    audio_path=AUDIO_PATH,
    out_path=OUTPUT_PATH,
    width=WIDTH, height=HEIGHT,
    crf=17, preset='medium',
    progress_every=120,
)
print('\nWrote:', OUTPUT_PATH)

## 4. Preview

Either open the file directly in your video player, or play it inline below.

In [ ]:
from IPython.display import Video
Video(OUTPUT_PATH, embed=False, width=720)